# Operational schema — related tables + keys (executed)

Runs against Lakebase `app` schema on the **development** branch and shows the modeled relational schema with its foreign keys and primary keys.

In [1]:
import psycopg2, os, subprocess, json
# OAuth token + host resolved from the Lakebase 'development' branch endpoint
host = subprocess.run(['databricks','postgres','list-endpoints',
    'projects/meridian-bank/branches/development','-p','fe-vm-serverless-stable-tech-summit','-o','json'],
    capture_output=True,text=True).stdout
host = json.loads(host)[0]['status']['hosts']['host']
tok = json.loads(subprocess.run(['databricks','postgres','generate-database-credential',
    'projects/meridian-bank/branches/development/endpoints/primary','-p','fe-vm-serverless-stable-tech-summit','-o','json'],
    capture_output=True,text=True).stdout)['token']
conn = psycopg2.connect(host=host, port=5432, dbname='databricks_postgres',
    user=os.environ['USER_EMAIL'], password=tok, sslmode='require')
cur = conn.cursor()
def run(sql):
    cur.execute(sql)
    for r in cur.fetchall(): print(' | '.join(str(x) for x in r))
print('connected')

connected


In [1]:
print("=== tables (writable vs read-only synced) ===")
run("""SELECT relname, CASE relkind WHEN 'r' THEN 'writable table' WHEN 'p' THEN 'read-only synced (partitioned)' END
       FROM pg_class c JOIN pg_namespace n ON n.oid=c.relnamespace
       WHERE n.nspname='app' AND relkind IN ('r','p') ORDER BY relkind, relname""")

customer_position|read-only synced (partitioned)
nba_recommendations|read-only synced (partitioned)
open_atrisk|read-only synced (partitioned)
products|read-only synced (partitioned)
conversations|writable table
feedback|writable table
messages|writable table
products_search|writable table
rm_actions|writable table


In [1]:
print("=== foreign keys (related tables) ===")
run("""SELECT tc.table_name||' . '||kcu.column_name||'  ->  '||ccu.table_name||' . '||ccu.column_name
       FROM information_schema.table_constraints tc
       JOIN information_schema.key_column_usage kcu ON tc.constraint_name=kcu.constraint_name
       JOIN information_schema.constraint_column_usage ccu ON ccu.constraint_name=tc.constraint_name
       WHERE tc.constraint_type='FOREIGN KEY' AND tc.table_schema='app'""")

messages . conversation_id  ->  conversations . id
feedback . message_id  ->  messages . id


In [1]:
print("=== primary keys ===")
run("""SELECT table_name||' (PK: '||string_agg(column_name,', ')||')'
       FROM information_schema.table_constraints tc JOIN information_schema.key_column_usage kcu USING(constraint_name)
       WHERE tc.constraint_type='PRIMARY KEY' AND tc.table_schema='app' GROUP BY table_name ORDER BY table_name""")

ERROR:  column reference "table_name" is ambiguous
LINE 1: SELECT table_name||' (PK: '||string_agg(column_name,', ')||'...
               ^
